In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

# Load DataFrame MC

In [ ]:
from cols_to_keep import *
keys2load = ["cc1pi", "nudf", "hdr", "histpotdf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_1e20_training.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100, filter_df = False, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']
mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
cols_to_keep = truth_cols_to_keep_slim
del mc_bnb_df
gc.collect()

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']
del data_df
gc.collect()

pot_weight_col = ('slc', 'wgt', '', '', '', '')
# BNB data
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))

# BNB MC
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

mc_evt_df = mc_bnb_evt_df
del mc_bnb_evt_df  # free original immediately — mc_evt_df holds the reference now
gc.collect()
mc_evt_df = perform_truth_matching(mc_evt_df, mc_bnb_nu_df)
del mc_bnb_nu_df

mc_cumulative_masks = build_event_cumulative_masks(mc_evt_df, sideband = "")
data_cumulative_masks = build_event_cumulative_masks(data_evt_df, sideband = "")

# Loop
for name in mc_cumulative_masks.keys():
    HelperFunctions.print_purity(mc_evt_df[mc_cumulative_masks[name].groupby(['__ntuple', 'entry', 'rec.slc..index']).first()], ('truth','nu_categ','','','',''))

# Loop
for name in mc_cumulative_masks.keys():
    n_mc   = get_n_evt(mc_evt_df,   mc_cumulative_masks[name],   use_weight=True)
    n_data = get_n_evt(data_evt_df, data_cumulative_masks[name], use_weight=False)
    print(f"{name:<15} | {n_mc:<12.2f} | {n_data:<12}")

In [ ]:
from cols_to_keep import *

# Files with the _update_cv suffix; all variables below carry the _ucv suffix
# so they can coexist with the original (non-updated) dataframes in the same session.
base_dir      = "/exp/sbnd/data/users/lpelegri/cafpyana_data"
file_tag_ucv  = "_update_cv"
mc_path_ucv   = f"{base_dir}/cc1pi_1e20_training{file_tag_ucv}.df"
data_path_ucv = f"{base_dir}/cc1pi_data_fixdev_bnblight{file_tag_ucv}.df"

# Load MC
keys2load_mc_ucv = ["cc1pi", "nudf", "hdr", "histpotdf"]  ## keys from the configuration file
mc_bnb_df_ucv = load_df(mc_path_ucv, keys2load_mc_ucv, 100, filter_df=False, reprocess_df=False, reprocess_truth=False)
mc_bnb_evt_df_ucv = mc_bnb_df_ucv['cc1pi']
mc_bnb_nu_df_ucv  = mc_bnb_df_ucv['nudf']
mc_bnb_hdr_df_ucv = mc_bnb_df_ucv['hdr']
cols_to_keep_ucv = truth_cols_to_keep_slim
del mc_bnb_df_ucv
gc.collect()

# Load data
keys2load_data_ucv = ["cc1pi", "hdr", "histpotdf"]  ## keys from the configuration file
data_df_ucv = load_df(data_path_ucv, keys2load_data_ucv, 100)
data_evt_df_ucv = data_df_ucv['cc1pi']
data_hdr_df_ucv = data_df_ucv['hdr']
del data_df_ucv
gc.collect()

pot_weight_col = ('slc', 'wgt', '', '', '', '')

# BNB data
data_tot_pot_ucv = data_hdr_df_ucv['pot'].sum()
print("data_tot_pot_ucv: %.3e" % (data_tot_pot_ucv))
data_evt_df_ucv[pot_weight_col] = np.ones(len(data_evt_df_ucv))

# BNB MC
mc_tot_pot_ucv = mc_bnb_hdr_df_ucv['pot'].sum()
print("mc_tot_pot_ucv: %.3e" % (mc_tot_pot_ucv))
mc_pot_scale_ucv = data_tot_pot_ucv / mc_tot_pot_ucv
print("mc_pot_scale_ucv: %.3e" % (mc_pot_scale_ucv))
mc_bnb_evt_df_ucv[pot_weight_col] = mc_pot_scale_ucv * np.ones(len(mc_bnb_evt_df_ucv))

mc_evt_df_ucv = mc_bnb_evt_df_ucv
del mc_bnb_evt_df_ucv  # mc_evt_df_ucv holds the reference now
gc.collect()
mc_evt_df_ucv = perform_truth_matching(mc_evt_df_ucv, mc_bnb_nu_df_ucv)
del mc_bnb_nu_df_ucv

mc_cumulative_masks_ucv   = build_event_cumulative_masks(mc_evt_df_ucv,   sideband="")
data_cumulative_masks_ucv = build_event_cumulative_masks(data_evt_df_ucv, sideband="")

# Purity after each cut
for name in mc_cumulative_masks_ucv.keys():
    HelperFunctions.print_purity(
        mc_evt_df_ucv[mc_cumulative_masks_ucv[name].groupby(['__ntuple', 'entry', 'rec.slc..index']).first()],
        ('truth', 'nu_categ', '', '', '', ''))

# MC vs data event counts after each cut
for name in mc_cumulative_masks_ucv.keys():
    n_mc_ucv   = get_n_evt(mc_evt_df_ucv,   mc_cumulative_masks_ucv[name],   use_weight=True)
    n_data_ucv = get_n_evt(data_evt_df_ucv, data_cumulative_masks_ucv[name], use_weight=False)
    print(f"{name:<15} | {n_mc_ucv:<12.2f} | {n_data_ucv:<12}")

# Chi2 comparisons

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

col_BDTG_score_proton = ('pfp', 'trk', 'bdt_proton_score', '', '', '')

diff_vars = {
    ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', ''):   r'$\chi^2_{\mu} - \chi^2_{\mu,\,\mathrm{ucv}}$',
    ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', ''): r'$\chi^2_{p} - \chi^2_{p,\,\mathrm{ucv}}$',
    ('pfp', 'trk', 'chi2_exp_pol', '', '', ''):           r'chi2_exp_pol $-$ chi2_exp_pol$_{\mathrm{ucv}}$',
    ('pfp', 'trk', 'frac50', '', '', ''):                 r'frac50 $-$ frac50$_{\mathrm{ucv}}$',
    col_BDTG_score_proton:                                r'BDT proton score $-$ BDT proton score$_{\mathrm{ucv}}$',
}

# Fixed (min, max) per variable; None -> symmetric range from the 1st-99th percentile of |difference|
diff_ranges = {col: None for col in diff_vars}
n_bins   = 50
pot_unit = 1e20
log_y    = False

# Selection: None -> all tracks; otherwise keep tracks passing this cut in BOTH versions
cut_name = None

def matched_diff(df_nom, df_ucv, col, mask_nom=None, mask_ucv=None):
    """Return nominal - ucv for tracks present in both dataframes, with the nominal POT weight."""
    x_nom = df_nom[col]
    x_ucv = df_ucv[col]
    w_nom = df_nom[pot_weight_col]
    if mask_nom is not None:
        x_nom, w_nom = x_nom[mask_nom], w_nom[mask_nom]
        x_ucv = x_ucv[mask_ucv]
    assert x_nom.index.is_unique and x_ucv.index.is_unique, "index does not identify tracks uniquely"
    common = x_nom.index.intersection(x_ucv.index)
    d = x_nom.loc[common].to_numpy(dtype=float) - x_ucv.loc[common].to_numpy(dtype=float)
    w = w_nom.loc[common].to_numpy(dtype=float)
    ok = np.isfinite(d)
    return d[ok], w[ok], len(x_nom), len(x_ucv), len(common)

def hist_with_err(x, w, edges):
    h,   _ = np.histogram(x, bins=edges, weights=w)
    sw2, _ = np.histogram(x, bins=edges, weights=w**2)
    return h, np.sqrt(sw2)

def normalize(h, err, mode, pot):
    scale = pot_unit / pot if mode == "pot" else (1.0 / h.max() if h.max() > 0 else 0.0)
    return h * scale, err * scale

# Selection: keep tracks passing the selection up to slc.cut.angle in BOTH versions
cut_col    = ('slc', 'cut', 'angle', '', '', '')
angle_keys = [k for k in mc_cumulative_masks if 'angle' in str(k).lower()]
cut_name   = angle_keys[0] if angle_keys else 'slc.cut.angle'
print("Cumulative-mask keys:", list(mc_cumulative_masks.keys()))
print("Using selection:", cut_name if angle_keys else f"column {cut_col}")

def sel_masks(kind):
    if angle_keys:
        if kind == "mc":
            return mc_cumulative_masks[cut_name], mc_cumulative_masks_ucv[cut_name]
        return data_cumulative_masks[cut_name], data_cumulative_masks_ucv[cut_name]
    df_nom, df_ucv = (mc_evt_df, mc_evt_df_ucv) if kind == "mc" else (data_evt_df, data_evt_df_ucv)
    return df_nom[cut_col].fillna(0).astype(bool), df_ucv[cut_col].fillna(0).astype(bool)
samples = {
    "MC":   dict(df_nom=mc_evt_df,   df_ucv=mc_evt_df_ucv,   masks=sel_masks("mc"),   color="C0"),
    "Data": dict(df_nom=data_evt_df, df_ucv=data_evt_df_ucv, masks=sel_masks("data"), color="k"),
}

modes = ["max", "pot"]
fig, axes = plt.subplots(len(diff_vars), 2, figsize=(13, 4.2 * len(diff_vars)), squeeze=False)

for i, (col, xlabel) in enumerate(diff_vars.items()):
    diffs = {}
    for tag, s in samples.items():
        d, w, n_nom, n_ucv, n_match = matched_diff(s["df_nom"], s["df_ucv"], col, *s["masks"])
        diffs[tag] = (d, w)
        print(f"{col[2:5]} {tag:<4} | matched {n_match}/{n_nom} (nom), {n_match}/{n_ucv} (ucv) | "
              f"identical: {np.mean(d == 0):.3f} | mean: {np.mean(d):.4g} | std: {np.std(d):.4g}")

    if diff_ranges[col] is not None:
        edges = np.linspace(*diff_ranges[col], n_bins + 1)
    else:
        a = np.percentile(np.abs(np.concatenate([d for d, _ in diffs.values()])), 99)
        a = a if a > 0 else 1e-3
        edges = np.linspace(-a, a, n_bins + 1)
    centers = 0.5 * (edges[1:] + edges[:-1])

    for j, mode in enumerate(modes):
        ax = axes[i, j]
        h_mc, e_mc = normalize(*hist_with_err(*diffs["MC"],   edges), mode, data_tot_pot)
        h_d,  e_d  = normalize(*hist_with_err(*diffs["Data"], edges), mode, data_tot_pot)

        ax.stairs(h_mc, edges, color=samples["MC"]["color"], lw=1.5, label="MC")
        ax.stairs(h_mc + e_mc, edges, baseline=h_mc - e_mc, fill=True,
                  color=samples["MC"]["color"], alpha=0.25)
        ax.errorbar(centers, h_d, yerr=e_d, fmt="o", color=samples["Data"]["color"], ms=4, label="Data")
        ax.axvline(0, color="gray", ls="--", lw=0.8)

        ax.set_xlim(edges[0], edges[-1])
        if log_y:
            ax.set_yscale("log")
        ax.set_xlabel(xlabel + "  (nominal $-$ update_cv)")
        ax.set_ylabel("Normalized to maximum" if mode == "max" else f"Entries / {pot_unit:.0e} POT")
    axes[i, 0].legend(fontsize=9)

axes[0, 0].set_title("Normalized to maximum")
axes[0, 1].set_title("Normalized to POT")
fig.suptitle(f"Nominal $-$ update_cv, selection: {cut_name if cut_name else 'none'}", y=1.0)
fig.tight_layout()
# fig.savefig("diff_nominal_minus_update_cv.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

var_labels = {
    ('pfp', 'trk', 'chi2pid', 'best', 'chi2_muon', ''):   r'$\chi^2_{\mu}$',
    ('pfp', 'trk', 'chi2pid', 'best', 'chi2_proton', ''): r'$\chi^2_{p}$',
    ('pfp', 'trk', 'chi2_exp_pol', '', '', ''):           'chi2_exp_pol',
    ('pfp', 'trk', 'frac50', '', '', ''):                 'frac50',
    col_BDTG_score_proton:                                'BDT proton score',
}

# Fixed (min, max) for both axes; None -> 1st-99th percentile of nominal and update_cv values combined
xy_ranges = {col: None for col in diff_vars}
n_bins = 50
log_z  = True

def matched_values(df_nom, df_ucv, col, mask_nom=None, mask_ucv=None):
    """Return nominal value, update_cv value, and nominal POT weight for tracks present in both versions."""
    x_nom = df_nom[col]
    x_ucv = df_ucv[col]
    w_nom = df_nom[pot_weight_col]
    if mask_nom is not None:
        x_nom, w_nom = x_nom[mask_nom], w_nom[mask_nom]
        x_ucv = x_ucv[mask_ucv]
    common = x_nom.index.intersection(x_ucv.index)
    xn = x_nom.loc[common].to_numpy(dtype=float)
    xu = x_ucv.loc[common].to_numpy(dtype=float)
    w  = w_nom.loc[common].to_numpy(dtype=float)
    ok = np.isfinite(xn) & np.isfinite(xu)
    return xn[ok], xu[ok], w[ok]

fig, axes = plt.subplots(len(diff_vars), 2, figsize=(11, 5 * len(diff_vars)),
                         squeeze=False, constrained_layout=True)

for i, col in enumerate(diff_vars):
    pairs = {tag: matched_values(s["df_nom"], s["df_ucv"], col, *s["masks"])
             for tag, s in samples.items()}

    if xy_ranges[col] is not None:
        lo, hi = xy_ranges[col]
    else:
        all_vals = np.concatenate([np.concatenate([xn, xu]) for xn, xu, _ in pairs.values()])
        lo, hi = np.percentile(all_vals, [1, 99])
    edges = np.linspace(lo, hi, n_bins + 1)

    hists = {}
    for tag, (xn, xu, w) in pairs.items():
        h, _, _ = np.histogram2d(xn, xu, bins=[edges, edges], weights=w)
        hists[tag] = h / h.sum() if h.sum() > 0 else h

    vmax = max(h.max() for h in hists.values())
    if log_z:
        vmin = min(h[h > 0].min() for h in hists.values() if (h > 0).any())
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = Normalize(vmin=0, vmax=vmax)

    for j, tag in enumerate(["Data", "MC"]):
        ax = axes[i, j]
        h = np.where(hists[tag] > 0, hists[tag], np.nan)
        mesh = ax.pcolormesh(edges, edges, h.T, norm=norm, cmap="viridis")
        ax.plot([lo, hi], [lo, hi], color="gray", ls="--", lw=0.8)
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.set_aspect("equal")
        ax.set_xlabel(f"{var_labels[col]} (nominal)")
        ax.set_ylabel(f"{var_labels[col]} (update_cv)")
        ax.set_title(tag)
    fig.colorbar(mesh, ax=axes[i, :], label="Fraction of entries", shrink=0.9)

fig.suptitle(f"update_cv vs. nominal, selection: {cut_name}")
# fig.savefig("ucv_vs_nominal_2d.pdf", bbox_inches="tight")
plt.show()